# Image Enhancement and Per-Image Selection (Phase 0)
**Team Triathlon**: Ananya Kasaudhan, Suyash Srivastav, Logi Vasan  
**Supervisor**: Prof. Kishor Upla, Department of Electronics Engineering, SVNIT Surat  
---
### Overview
This notebook explores per-image enhancement selection formulated as an algorithm selection problem (Rice, 1976).
We use 7 cheap handcrafted physical statistics from degraded images to predict the PSNR-optimal enhancer from a 6-method bank.

In [ ]:
import sys
sys.path.append("..")
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.methods import METHODS
from src.features import extract_features, FEATURE_NAMES
from src.labels import compute_psnr, compute_ssim, score_pair
from src.dataset import load_and_preprocess_pair

print("Libraries and project modules successfully loaded!")

## 1. Inspecting the Six Enhancement Methods on an Example Image

In [ ]:
low_path = "../data/lol_dataset/eval15/low/1.png"
high_path = "../data/lol_dataset/eval15/high/1.png"
clean, deg, scale = load_and_preprocess_pair(low_path, high_path, target_long_side=512)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

axes[0].imshow(cv2.cvtColor(deg, cv2.COLOR_BGR2RGB))
axes[0].set_title("Degraded Input (Low-Light)")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(clean, cv2.COLOR_BGR2RGB))
axes[1].set_title("Ground Truth Reference")
axes[1].axis("off")

for idx, (m_name, m_fn) in enumerate(METHODS.items()):
    out = m_fn(deg)
    p = compute_psnr(clean, out)
    s = compute_ssim(clean, out)
    axes[idx + 2].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    axes[idx + 2].set_title(f"{m_name}
PSNR: {p:.2f} dB | SSIM: {s:.3f}")
    axes[idx + 2].axis("off")

plt.tight_layout()
plt.show()

## 2. Extracting Handcrafted Physical Features

In [ ]:
feats = extract_features(deg, include_engineered=True)
df_f = pd.DataFrame([feats])
print("Degraded Image Handcrafted Physical Descriptors:")
display(df_f.T.rename(columns={0: "Value"}))

## 3. Loading Benchmark Results & Regret Tables

In [ ]:
df_summary = pd.read_csv("../results/metrics_table.csv")
print("=== Primary Benchmark Results (GroupKFold CV) ===")
display(df_summary)

df_eval15 = pd.read_csv("../results/eval15_test_results.csv")
print("=== Official eval15 Test Results ===")
display(df_eval15)

## 4. Visualizing Model Evaluation Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 13))

img1 = cv2.imread("../results/figures/fig1_oracle_distribution.png")
axes[0, 0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("Figure 1: Oracle Method Distribution")
axes[0, 0].axis("off")

img2 = cv2.imread("../results/figures/fig2_confusion_matrix.png")
axes[0, 1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title("Figure 2: Confusion Matrix (Random Forest)")
axes[0, 1].axis("off")

img3 = cv2.imread("../results/figures/fig3_regret_cdf.png")
axes[1, 0].imshow(cv2.cvtColor(img3, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("Figure 3: Empirical CDF of Regret (dB)")
axes[1, 0].axis("off")

img4 = cv2.imread("../results/figures/fig4_feature_importances.png")
axes[1, 1].imshow(cv2.cvtColor(img4, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("Figure 4: Feature Importances")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 5. Statistical Significance Testing
Paired two-tailed t-tests, Wilcoxon signed-rank tests, and 95% confidence intervals across all 15 grouped cross-validation folds:

In [ ]:
df_stats = pd.read_csv("../results/statistical_tests.csv")
print("=== Statistical Significance Testing (RF vs. Baselines across 15 Runs) ===")
display(df_stats)

## 6. Qualitative Restoration Comparison Strip (Figure 5)
Contrasts degraded inputs, the best fixed method (Retinex), the proposed selector output, and ground truth across diverse scenes:

In [ ]:
plt.figure(figsize=(18, 12))
img5 = cv2.imread("../results/figures/fig5_qualitative_strip.png")
plt.imshow(cv2.cvtColor(img5, cv2.COLOR_BGR2RGB))
plt.title("Figure 5: Qualitative Restoration Comparison Strip (LOL v1 Benchmark)", fontsize=14, fontweight="bold")
plt.axis("off")
plt.show()

## 7. Feature Ablation Study
Impact of removing individual feature groups on Top-1 Accuracy and Decision Regret:

In [ ]:
df_ablation = pd.read_csv("../results/ablation_table.csv")
display(df_ablation)